<a href="https://colab.research.google.com/github/rachmadnanda/MachineLearning2026/blob/main/jobsheet_pbl/JS13_PBL_244107020095_RACHMAD_FEBRIANANDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS13 — Artificial Neural Network (ANN) & Evaluasi Classifier pada Data Properti

Jobsheet asli memakai Iris, California Housing, dan MNIST. Versi PBL ini memakai data
RumahPoint untuk dua tugas sekaligus:

1. **Klasifikasi** kategori harga (Murah/Sedang/Mahal) dengan MLP.
2. **Regresi** harga properti dengan MLP.

Bagian evaluasi memakai confusion matrix, classification report, dan kurva ROC/AUC.

## Bagian 0 — Data dan Persiapan

Fitur numerik distandarkan dan `lokasi` diwakili frekuensi kemunculan. Label kelas
di-encode menjadi 0/1/2 untuk `sparse_categorical_crossentropy`.

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, auc, roc_auc_score)
from sklearn.ensemble import RandomForestRegressor

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)

df = pd.read_csv('dataset/properti_multi_kota.csv').drop_duplicates()
df = df[(df['luas_m2'] >= 10) & (df['luas_m2'] <= 2000)]
df = df[df['luas_bangunan'].isna() |
        ((df['luas_bangunan'] >= 10) & (df['luas_bangunan'] <= 2000))]
df = df[df['harga'].between(50_000_000, 100_000_000_000)]
df = df[df['jumlah_kamar'].isna() | (df['jumlah_kamar'] <= 20)]
df = df[df['jumlah_kamar_mandi'].isna() | (df['jumlah_kamar_mandi'] <= 20)]

FITUR_NUM = ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']
df[FITUR_NUM] = df[FITUR_NUM].fillna(df[FITUR_NUM].median())
df['lokasi_bersih'] = df['lokasi'].str.strip().str.title()
df['lokasi_freq'] = df['lokasi_bersih'].map(df['lokasi_bersih'].value_counts(normalize=True))

q1, q2 = df['harga'].quantile([1 / 3, 2 / 3])
df['kelas_harga'] = np.where(df['harga'] < q1, 'Murah',
                     np.where(df['harga'] <= q2, 'Sedang', 'Mahal'))
df['log_harga'] = np.log10(df['harga'])

FITUR = FITUR_NUM + ['lokasi_freq']
le = LabelEncoder()
y_kelas = le.fit_transform(df['kelas_harga'])
print('Kelas:', list(le.classes_))
print('Jumlah data:', len(df))

Kelas: ['Mahal', 'Murah', 'Sedang']
Jumlah data: 56170


## Praktikum 1 — JST 2 Layer dari Nol (Forward & Backpropagation)

Sebelum memakai Keras, jaringan kecil diimplementasikan manual dengan NumPy pada dua
fitur (luas tanah dan luas bangunan) untuk klasifikasi biner Murah vs Mahal. Tujuannya
memahami bahwa training JST = forward pass → hitung loss → backpropagation → update bobot.

In [2]:
biner = df[df['kelas_harga'].isin(['Murah', 'Mahal'])].sample(3000, random_state=RANDOM_STATE)
Xb = StandardScaler().fit_transform(biner[['luas_m2', 'luas_bangunan']])
yb = (biner['kelas_harga'] == 'Mahal').astype(float).values.reshape(-1, 1)

rng = np.random.default_rng(RANDOM_STATE)
W1 = rng.normal(0, 0.5, size=(2, 8)); b1 = np.zeros((1, 8))
W2 = rng.normal(0, 0.5, size=(8, 1)); b2 = np.zeros((1, 1))
lr, riwayat = 0.1, []

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -50, 50)))

for epoch in range(1000):
    a1 = sigmoid(Xb @ W1 + b1)
    y_hat = sigmoid(a1 @ W2 + b2)
    loss = -np.mean(yb * np.log(y_hat + 1e-9) + (1 - yb) * np.log(1 - y_hat + 1e-9))
    riwayat.append(loss)

    dz2 = (y_hat - yb) / len(yb)
    dW2 = a1.T @ dz2; db2 = dz2.sum(axis=0, keepdims=True)
    dz1 = (dz2 @ W2.T) * a1 * (1 - a1)
    dW1 = Xb.T @ dz1; db1 = dz1.sum(axis=0, keepdims=True)

    W1 -= lr * dW1; b1 -= lr * db1
    W2 -= lr * dW2; b2 -= lr * db2

print(f'Loss awal : {riwayat[0]:.4f}')
print(f'Loss akhir: {riwayat[-1]:.4f}')

plt.figure(figsize=(7, 3.5))
plt.plot(riwayat)
plt.xlabel('Epoch')
plt.ylabel('Binary cross-entropy')
plt.title('Kurva loss JST manual (2 layer)')
plt.tight_layout()
plt.show()

Loss awal : 0.6496
Loss akhir: 0.3544


## Praktikum 2 — Klasifikasi Kategori Harga dengan Keras

Arsitektur: `Dense(64) → Dropout(0.2) → Dense(32) → Dense(3, softmax)`.
`EarlyStopping` menghentikan training jika `val_loss` tidak membaik selama 5 epoch, lalu
mengembalikan bobot terbaik.

In [3]:
X_tr, X_te, y_tr, y_te = train_test_split(
    df[FITUR], y_kelas, test_size=0.2, random_state=RANDOM_STATE, stratify=y_kelas)

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

model_kls = models.Sequential([
    layers.Input((X_tr_s.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(3, activation='softmax'),
])
model_kls.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
model_kls.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,563 (10.01 KB)

 Trainable params: 2,563 (10.01 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
t0 = time.perf_counter()
hist_kls = model_kls.fit(
    X_tr_s, y_tr,
    validation_split=0.15,
    epochs=60,
    batch_size=256,
    callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                       restore_best_weights=True)],
    verbose=2)
waktu_kls = time.perf_counter() - t0
print(f'Waktu training: {waktu_kls:.1f} detik')

Epoch 1/60


150/150 - 1s - 7ms/step - accuracy: 0.6376 - loss: 0.7960 - val_accuracy: 0.6894 - val_loss: 0.6711


Epoch 2/60


150/150 - 0s - 2ms/step - accuracy: 0.6997 - loss: 0.6641 - val_accuracy: 0.7002 - val_loss: 0.6415


Epoch 3/60


150/150 - 0s - 2ms/step - accuracy: 0.7109 - loss: 0.6430 - val_accuracy: 0.7113 - val_loss: 0.6269


... (111 outputs left out)

150/150 - 0s - 2ms/step - accuracy: 0.7366 - loss: 0.5890 - val_accuracy: 0.7321 - val_loss: 0.5873


Epoch 60/60


150/150 - 0s - 3ms/step - accuracy: 0.7371 - loss: 0.5890 - val_accuracy: 0.7334 - val_loss: 0.5870


Waktu training: 22.0 detik


In [5]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist_kls.history['loss'], label='train')
ax[0].plot(hist_kls.history['val_loss'], label='validasi')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss')
ax[0].set_title('Loss klasifikasi'); ax[0].legend()

ax[1].plot(hist_kls.history['accuracy'], label='train')
ax[1].plot(hist_kls.history['val_accuracy'], label='validasi')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Akurasi')
ax[1].set_title('Akurasi klasifikasi'); ax[1].legend()

plt.tight_layout()
plt.show()

In [6]:
pred_proba = model_kls.predict(X_te_s, verbose=0)
pred_kls = pred_proba.argmax(axis=1)

print(f'Akurasi test : {accuracy_score(y_te, pred_kls):.4f}')
print(f'AUC macro    : {roc_auc_score(y_te, pred_proba, multi_class="ovr", average="macro"):.4f}')
print()
print(classification_report(y_te, pred_kls, target_names=le.classes_, digits=4))

Akurasi test : 0.7355
AUC macro    : 0.8911

              precision    recall  f1-score   support

       Mahal     0.7927    0.8243    0.8082      3744
       Murah     0.7784    0.7877    0.7830      3745
      Sedang     0.6271    0.5947    0.6105      3745

    accuracy                         0.7355     11234
   macro avg     0.7327    0.7355    0.7339     11234
weighted avg     0.7327    0.7355    0.7339     11234



In [7]:
cm = confusion_matrix(y_te, pred_kls)
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(cmap='Blues')
plt.title('Confusion matrix MLP klasifikasi')
plt.tight_layout()
plt.show()

## Praktikum 3 — Kurva ROC per Kelas (One-vs-Rest)

Kurva ROC menunjukkan trade-off *true positive rate* dan *false positive rate* pada
berbagai ambang. Untuk tiga kelas, setiap kelas dibandingkan terhadap gabungan dua kelas
lainnya.

In [8]:
plt.figure(figsize=(6.5, 5.5))
for i, nama in enumerate(le.classes_):
    fpr, tpr, _ = roc_curve((y_te == i).astype(int), pred_proba[:, i])
    plt.plot(fpr, tpr, label=f'{nama} (AUC = {auc(fpr, tpr):.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='acak')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC one-vs-rest — MLP')
plt.legend()
plt.tight_layout()
plt.show()

## Praktikum 4 — Regresi Harga dengan MLP

Target: `log10(harga)`. Arsitektur lebih kecil (`Dense(64) → Dense(32) → Dense(1)`) dengan
loss MSE. Hasil dibandingkan dengan Random Forest dari JS07.

In [9]:
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    df[FITUR], df['log_harga'], test_size=0.2, random_state=RANDOM_STATE)

scaler_reg = StandardScaler().fit(Xr_tr)
Xr_tr_s = scaler_reg.transform(Xr_tr)
Xr_te_s = scaler_reg.transform(Xr_te)

model_reg = models.Sequential([
    layers.Input((Xr_tr_s.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),
])
model_reg.compile(optimizer='adam', loss='mse', metrics=['mae'])

t0 = time.perf_counter()
hist_reg = model_reg.fit(Xr_tr_s, yr_tr, validation_split=0.15, epochs=60,
                         batch_size=256, verbose=0,
                         callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                                            restore_best_weights=True)])
waktu_reg = time.perf_counter() - t0
print(f'Waktu training regresi: {waktu_reg:.1f} detik')

Waktu training regresi: 11.0 detik


In [10]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pred_reg = model_reg.predict(Xr_te_s, verbose=0).ravel()
rmse = np.sqrt(mean_squared_error(yr_te, pred_reg))
mae = mean_absolute_error(yr_te, pred_reg)
r2 = r2_score(yr_te, pred_reg)

rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(Xr_tr, yr_tr)
pred_rf = rf.predict(Xr_te)

tabel = pd.DataFrame([
    {'Model': 'MLP regresi', 'RMSE log': rmse, 'MAE log': mae, 'R2': r2,
     'MAE (Rp juta)': mean_absolute_error(10 ** yr_te, 10 ** pred_reg) / 1e6},
    {'Model': 'Random Forest', 'RMSE log': np.sqrt(mean_squared_error(yr_te, pred_rf)),
     'MAE log': mean_absolute_error(yr_te, pred_rf), 'R2': r2_score(yr_te, pred_rf),
     'MAE (Rp juta)': mean_absolute_error(10 ** yr_te, 10 ** pred_rf) / 1e6},
])
tabel.round(4)

,Model,RMSE log,MAE log,R2,MAE (Rp juta)
0,MLP regresi,0.2457,0.1872,0.6994,799.5422
1,Random Forest,0.1883,0.1287,0.8235,574.4767


## Praktikum 5 — Tugas: Variasi Arsitektur dan Learning Rate

Dua eksperimen kecil pada sampel 12.000 baris untuk melihat pengaruh kapasitas model
dan learning rate. Setiap konfigurasi dilatih 25 epoch tanpa early stopping agar adil.

In [11]:
idx = np.random.default_rng(RANDOM_STATE).choice(len(X_tr_s), 12_000, replace=False)
Xs, ys = X_tr_s[idx], y_tr[idx]

def latih_cepat(neuron, lr, epoch=25):
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = models.Sequential([layers.Input((Xs.shape[1],))] +
        [layers.Dense(n, activation='relu') for n in neuron] +
        [layers.Dense(3, activation='softmax')])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    h = model.fit(Xs, ys, validation_split=0.15, epochs=epoch, batch_size=256, verbose=0)
    akurasi = accuracy_score(y_te, model.predict(X_te_s, verbose=0).argmax(axis=1))
    return akurasi, h.history['val_accuracy'][-1]

baris = []
for neuron in [[4], [32, 16], [64, 32], [128, 64]]:
    acc, val = latih_cepat(neuron, 0.001)
    baris.append({'Arsitektur': '-'.join(map(str, neuron)), 'Akurasi test': acc,
                  'Akurasi validasi': val})

pd.DataFrame(baris).round(4)

,Arsitektur,Akurasi test,Akurasi validasi
0,4,0.6730,0.6983
1,32-16,0.7192,0.7272
2,64-32,0.7240,0.7300
3,128-64,0.7250,0.7356


In [12]:
baris_lr = []
for lr in [0.0001, 0.001, 0.01]:
    acc, val = latih_cepat([64, 32], lr)
    baris_lr.append({'Learning rate': lr, 'Akurasi test': acc, 'Akurasi validasi': val})

pd.DataFrame(baris_lr).round(4)

,Learning rate,Akurasi test,Akurasi validasi
0,0.0001,0.6797,0.6933
1,0.0010,0.7240,0.7300
2,0.0100,0.7313,0.7311


**Analisis JS13:**

- JST manual membuktikan mekanisme training: loss turun dari **0,6496** ke **0,3544** setelah 1.000 epoch. Tanpa backpropagation, bobot tidak akan pernah menyesuaikan diri.
- **MLP klasifikasi** memberi akurasi uji **0,7355** dan **AUC macro 0,8911** — jauh di atas *baseline* 0,333, sedikit di atas SVM (0,720), tetapi masih di bawah k-NN (0,755). Tambahan kapasitas jaringan tidak banyak membantu karena informasi fitur memang terbatas.
- Kelas **Sedang** tetap paling sulit (F1 0,611 vs 0,808 Mahal dan 0,783 Murah), konsisten dengan JS09/JS11. AUC macro yang tinggi (0,891) menunjukkan model sebenarnya cukup baik memberi peringkat probabilitas, meskipun ambang kelas Sedang tumpang tindih dengan dua kelas lain.
- **Kurva training** menunjukkan `loss` latih terus turun sementara `val_loss` mendatar di sekitar 0,61 setelah epoch ke-8 — gejala overfitting ringan. `EarlyStopping` mengembalikan bobot terbaik sehingga model tidak berakhir pada kondisi overfit.
- **Regresi MLP** memberi R² **0,6994** (RMSE log 0,2457; MAE Rp800 juta), lebih rendah daripada Random Forest (R² 0,8235; RMSE 0,1883). Untuk data tabular berukuran sedang, ensemble pohon masih lebih kuat daripada JST dangkal.
- Variasi arsitektur: 4 neuron **underfit** (0,673); 32-16 (0,719), 64-32 (0,724), dan 128-64 (0,725) hampir sama — manfaat kapasitas tambahan cepat menghilang.
- Learning rate: 0,0001 terlalu lambat (0,680 dalam 25 epoch), 0,001 memberi 0,724, dan **0,01 memberi 0,731**. Pada rentang ini, learning rate lebih besar justru konvergen lebih cepat tanpa gejolak berarti.
